In [ ]:
#| default_exp topology

In [ ]:
#| export

from __future__ import annotations

import math
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Dict, Iterable, List, Mapping, Sequence, Tuple

from comm_core.core import NodeId


@dataclass
class Link:
    sender: NodeId
    receiver: NodeId

    bandwidth_hz: float = float("inf")
    frequency_hz: float = 0.0
    tx_power_w: float = 1.0

    enabled: bool = True

In [ ]:
#| export
class Topology(ABC):

    @abstractmethod
    def neighbors(self, node: NodeId) -> List[NodeId]:
        pass

    @abstractmethod
    def is_connected(
        self,
        sender: NodeId,
        receiver: NodeId,
    ) -> bool:
        pass

    @abstractmethod
    def get_link(
        self,
        sender: NodeId,
        receiver: NodeId,
    ) -> Link:
        pass



In [ ]:
#| export
class GraphTopology(Topology):

    def __init__(
        self,
        links: Iterable[Link],
    ):
        self._links: Dict[Tuple[NodeId, NodeId], Link] = {
            (link.sender, link.receiver): link
            for link in links
        }

    def neighbors(self, node: NodeId) -> List[NodeId]:
        return [
            receiver
            for (sender, receiver), link in self._links.items()
            if sender == node and link.enabled
        ]

    def is_connected(
        self,
        sender: NodeId,
        receiver: NodeId,
    ) -> bool:
        link = self._links.get((sender, receiver))
        return link is not None and link.enabled

    def get_link(
        self,
        sender: NodeId,
        receiver: NodeId,
    ) -> Link:
        try:
            return self._links[(sender, receiver)]
        except KeyError:
            raise ValueError(
                f"No link from {sender} to {receiver}"
            )



In [ ]:
#| export

class FullMeshTopology(GraphTopology):

    def __init__(
        self,
        nodes: Iterable[NodeId],
    ):
        nodes = list(nodes)

        links = [
            Link(sender=a, receiver=b)
            for a in nodes
            for b in nodes
            if a != b
        ]

        super().__init__(links)



In [ ]:
#| export

class StarTopology(GraphTopology):

    def __init__(
        self,
        center: NodeId,
        nodes: Iterable[NodeId],
    ):
        links = []

        for node in nodes:
            if node == center:
                continue

            links.append(
                Link(center, node)
            )
            links.append(
                Link(node, center)
            )

        super().__init__(links)

In [ ]:
#| export

def distance(
    a: Sequence[float], # Position of a node
    b: Sequence[float], # Position of another node
    ord: float = 2, # 2: Euclidean, 1: Manhattan, math.inf: Chebyshev (squares on a grid)
) -> float:
    "Distance between two positions."
    d = [abs(x - y) for x, y in zip(a, b)]
    if ord == math.inf:
        return max(d, default=0.0)
    if ord == 1:
        return sum(d)
    if ord == 2:
        return math.sqrt(sum(x * x for x in d))
    raise ValueError(f"ord: 1, 2 or math.inf, not {ord}")


class DistanceTopology(GraphTopology):
    """
    Links between the nodes at most `radius` apart (e.g. a radio range).
    Build a new one when the nodes move.
    """

    def __init__(
        self,
        positions: Mapping[NodeId, Sequence[float]], # Position of every node
        radius: float, # Largest distance of a link
        ord: float = 2, # Norm of the distance (see `distance`)
        **link_kwargs, # Properties of the links (e.g. `bandwidth_hz`)
    ):
        ids = list(positions)
        links = [
            Link(sender=a, receiver=b, **link_kwargs)
            for a in ids
            for b in ids
            if a != b and distance(positions[a], positions[b], ord) <= radius
        ]
        super().__init__(links)

### Tests

In [ ]:
mesh = FullMeshTopology(['a', 'b', 'c'])
assert sorted(mesh.neighbors('a')) == ['b', 'c'] and mesh.is_connected('c', 'b')
star = StarTopology('a', ['a', 'b', 'c'])
assert star.neighbors('b') == ['a'] and not star.is_connected('b', 'c')
near = DistanceTopology({'a': (0, 0), 'b': (1, 1), 'c': (5, 0)}, radius=1.5)
assert near.neighbors('a') == ['b'] and near.neighbors('c') == []
grid = DistanceTopology({'a': (0, 0), 'b': (1, 1)}, radius=1, ord=math.inf, bandwidth_hz=1e6)
assert grid.is_connected('a', 'b') and grid.get_link('a', 'b').bandwidth_hz == 1e6

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()